# TP 4 — Dask, un an de végétation et les changements autour de Taï

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/curat/data-science/notebooks/TP4_Dask_NDVI_Tai.ipynb)

**Data Sciences, visualisation et Big Data** · M2 Télédétection et SIG ·
Séance 4 · KOUADIO Kouao Laurent

Objectifs :
1. Mesurer le gain du calcul parallèle avec **Dask**.
2. Construire un **cube Sentinel-2 paresseux** avec `odc-stac`.
3. Calculer un **NDVI mensuel** et des **composites** sans nuages.
4. Cartographier les **baisses de NDVI** entre 2019 et 2025.

Durée : environ 90 minutes. Connexion Internet nécessaire ; les
lectures d'images prennent plusieurs minutes.

In [ ]:
# Google Colab : installe les paquets du cours (environ 1 minute).
# Dans l'environnement conda du cours (environment.yml), rien à faire.
import sys
if "google.colab" in sys.modules:
    %pip install -q pystac-client odc-stac

## 1. Imports

In [ ]:
import time

import dask
import dask.array as da
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from odc.stac import configure_rio, load
from pystac_client import Client

# relancer les lectures HTTP interrompues
configure_rio(cloud_defaults=True, GDAL_HTTP_MAX_RETRY="10",
              GDAL_HTTP_RETRY_DELAY="2")

## 2. Dask : chunks, graphe et parallélisme

Un tableau de 10 000 × 10 000 réels (0,8 Go), découpé en chunks.

In [ ]:
x = da.random.RandomState(0).random((10_000, 10_000),
                                     chunks=(1_000, 1_000))
y = (x - x.mean(axis=0)).std()
print(x.nbytes / 1e9, "Go,", x.numblocks, "chunks")
print(len(y.__dask_graph__()), "tâches dans le graphe")
y  # rien n'est encore calculé

In [ ]:
temps = {}
for n in (1, 2, 4, 8):
    t0 = time.time()
    y.compute(scheduler="threads", num_workers=n)
    temps[n] = time.time() - t0
    print(f"{n} thread(s) : {temps[n]:.2f} s")

plt.bar([str(n) for n in temps], temps.values(), color="#2D6A4F")
plt.xlabel("threads")
plt.ylabel("temps (s)")
plt.show()

**Question 1.** Quel est le gain avec 8 threads ? Combien de cœurs
physiques a votre machine (`os.cpu_count()` donne les threads) ?
Pourquoi le gain n'est-il pas égal au nombre de threads ?

## 3. Un cube Sentinel-2 paresseux

**Attention à l'offset.** Depuis 2022 (baseline ESA 04.00), ESA
ajoute 1 000 aux valeurs. Dans la collection `sentinel-2-l2a`
d'Earth Search, ce décalage est souvent déjà retiré
(`earthsearch:boa_offset_applied`). La fonction suivante donne le bon
offset pour chaque scène.

In [ ]:
cat = Client.open("https://earth-search.aws.element84.com/v1")
CLAIRS = [4, 5, 6, 11]  # végétation, sol nu, eau, neige (SCL)


def decalage(item):
    p = item.properties
    decale = (p.get("s2:processing_baseline", "00.00") >= "04.00"
              and not p.get("earthsearch:boa_offset_applied", False))
    return -0.1 if decale else 0.0


def cube_ndvi(bbox, dates, crs, resolution, max_nuages=80):
    items = cat.search(collections=["sentinel-2-l2a"], bbox=bbox,
                       datetime=dates,
                       query={"eo:cloud_cover": {"lt": max_nuages}}
                       ).item_collection()
    off = {i.datetime.strftime("%Y-%m-%d"): decalage(i) for i in items}
    ds = load(items, bands=["red", "nir", "scl"], bbox=bbox, crs=crs,
              resolution=resolution, chunks={}, groupby="solar_day")
    o = xr.DataArray([off[str(t)[:10]] for t in ds.time.values],
                     dims="time", coords={"time": ds.time})
    ok = ds["scl"].isin(CLAIRS)
    red = ds["red"].where(ok) * 1e-4 + o
    nir = ds["nir"].where(ok) * 1e-4 + o
    ndvi = (nir - red) / (nir + red)
    return ndvi.where(abs(ndvi) <= 1), len(items)


ndvi_tai, n = cube_ndvi([-7.20, 5.70, -7.15, 5.75],
                        "2025-01-01/2025-12-31", "EPSG:32629", 60)
print(n, "scènes")
ndvi_tai  # un DataArray dask : rien n'est encore lu

**Question 2.** Quelles sont les dimensions du cube ? Combien de
chunks contient-il (`ndvi_tai.data.numblocks`) ?

## 4. NDVI mensuel : forêt du Sud, savane du Nord

In [ ]:
sites = {"Sud (forêt, Taï)": ndvi_tai}
ndvi_nord, n = cube_ndvi([-5.55, 9.25, -5.50, 9.30],
                         "2025-01-01/2025-12-31", "EPSG:32630", 60)
sites["Nord (savane, Korhogo)"] = ndvi_nord

fig, ax = plt.subplots(figsize=(8, 4))
for nom, cube in sites.items():
    t0 = time.time()
    par_date = cube.median(["x", "y"])
    clair = cube.notnull().mean(["x", "y"])
    s = par_date.where(clair > 0.5).compute().to_series().dropna()
    mensuel = s.groupby(s.index.month).median()
    print(f"{nom} : {s.size} dates claires, {time.time() - t0:.0f} s")
    ax.plot(mensuel.index, mensuel.values, "o-", label=nom)
ax.set_xticks(range(1, 13))
ax.set_xlabel("mois")
ax.set_ylabel("NDVI médian")
ax.set_ylim(0, 1)
ax.legend()
plt.show()

**Question 3.** Décrivez les deux courbes. Pourquoi manque-t-il des
mois pour la forêt ? Reliez la courbe de la savane aux saisons.

## 5. Composites de saison sèche 2019 et 2025

Zone de 100 km × 100 km autour du massif de Taï, pixels de 100 m.

In [ ]:
bbox = [-7.70, 5.40, -6.80, 6.30]
comp = {}
for an in (2019, 2025):
    t0 = time.time()
    cube, n = cube_ndvi(bbox, f"{an}-01-01/{an}-02-28", "EPSG:32629",
                        100, max_nuages=60)
    comp[an] = cube.median("time").compute()
    print(f"{an} : {n} scènes, {time.time() - t0:.0f} s")

In [ ]:
d = comp[2025] - comp[2019]
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, an in zip(axs, (2019, 2025)):
    comp[an].plot.imshow(ax=ax, cmap="YlGn", vmin=0, vmax=0.9)
    ax.set_title(f"NDVI médian janv.-févr. {an}")
d.plot.imshow(ax=axs[2], cmap="RdBu", vmin=-0.4, vmax=0.4)
axs[2].set_title("Différence 2025 - 2019")
plt.show()

## 6. Surface des baisses de NDVI

In [ ]:
valide = d.notnull()
perte = (d < -0.2) & valide
print(f"pixels valides : {100 * float(valide.mean()):.1f} %")
print(f"baisse > 0,2 : {float(perte.sum()) * 0.01:.0f} km² "
      f"({100 * float(perte.sum() / valide.sum()):.1f} % des pixels)")

**Question 4.** Où se trouvent les fortes baisses ? Le massif
forestier central change-t-il ? Proposez deux causes de baisse de
NDVI qui ne sont **pas** une déforestation.

**Question 5 (bonus).** Recommencez avec un seuil de -0,3, puis avec
les mois de décembre-janvier. La surface change-t-elle beaucoup ?
Que concluez-vous sur la robustesse du résultat ?

## À rendre

Le notebook exécuté avec vos réponses aux questions 1 à 5.